# Ensemble experiment

This notebook combines the four frozen model outputs using three separately reported methods: majority/confidence span fusion, dev-weighted voting, and an optional LLM judge/refiner. The test set must not be used to derive weights.

Results, runtime, memory, and judge observations are intentionally blank until execution.

In [ ]:
from pathlib import Path
import json, sys
BASE = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd() / 'experimental_setup'
sys.path.insert(0, str(BASE))
from src.dataset import prepare_echr_files, load_prepared
from src.ensemble import majority_fusion, weighted_fusion, derive_weights
from src.metrics import score

DATA_ROOT = Path(r'E:\Docs\intern\AAintern\legal\experimental_setup\mining-legal-arguments')
PREPARED_ROOT = BASE / 'processed_echr'
MANIFEST = prepare_echr_files(DATA_ROOT, PREPARED_ROOT)
OUTPUT_DIR = Path('outputs')
MODEL_FILES = {
    'saul_7b': OUTPUT_DIR / '01_saul_7b_instruct.json',
    'saullm_54b': OUTPUT_DIR / '02_saullm_54b_instruct.json',
    'law_chat': OUTPUT_DIR / '03_law_chat.json',
    'llama3_8b': OUTPUT_DIR / '04_paper_llama3_8b_instruct.json',
}
EXPERIMENT_RECORD = {'date':'', 'dev_weights':'', 'judge_model':'', 'runtime_seconds':'', 'metrics':'', 'errors':'', 'observations':''}

def load_predictions(path):
    data=json.loads(Path(path).read_text(encoding='utf-8'))
    # The individual runner audit stores parsed output; rerun/export aligned predictions if needed.
    return data

from src.ensemble import load_prediction_file
test_records = load_prepared(PREPARED_ROOT, split='test')
test_preds = {name: load_prediction_file(path) for name, path in MODEL_FILES.items()}
# Method 1: majority span/label fusion.
majority = [majority_fusion({name: preds[i] for name, preds in test_preds.items()}) for i in range(len(test_records))]
majority_metrics = score(test_records, majority)

# Method 2: obtain weights from separate dev outputs, then apply them to test.
DEV_FILES = {name: path.with_name(path.stem + '_dev.json') for name, path in MODEL_FILES.items()}
dev_records = load_prepared(PREPARED_ROOT, split='dev')
dev_preds = {name: load_prediction_file(path) for name, path in DEV_FILES.items()}
weights = derive_weights(dev_records, dev_preds)
weighted = [weighted_fusion({name: preds[i] for name, preds in test_preds.items()}, weights) for i in range(len(test_records))]
weighted_metrics = score(test_records, weighted)

# Method 3 is intentionally a separately audited judge/refiner run. Feed it each input
# and the normalized candidates, then save aligned predictions as judge_predictions.json.
judge_metrics = ''  # fill after the judge/refiner has been run
print({'majority': majority_metrics, 'weighted': weighted_metrics, 'judge': judge_metrics, 'weights': weights})
